# Predator-prey: mass-action-restricted LASSO (sparse-learning-CRN)

This notebook re-runs the sparse-learning-CRN LASSO
(https://github.com/zwpku/sparse-learning-CRN) on the predator-prey data (example 4, $T=40$, $\lambda=0.01$),
**restricted to mass-action propensities**. Everything is done through inputs their code already accepts.
**Their C++ code is not modified.**

### What is different from the standard run (`sparse_runs/example4_T40_lam0p01`)

| | Standard run | This run |
|---|---|---|
| `xx_basis_flag` (config option) | 1 → quadratic basis is $x^2$ | **0** → quadratic basis is $x(x-1)$ |
| Basis functions per channel | all 6: $1, A, B, A^2, AB, B^2$ | only the mass-action candidates for that channel (the same candidate reactions used in the Bayesian analysis) |

The basis list is restricted by editing `output/basis_of_channel_<i>.txt` between `prepare` and `sparse_learning`,
which is the step their own instructions describe (`examples/HowToRun`, step 3).

### What their code still does *not* enforce
Their proximal step is an ordinary soft-threshold, so coefficients **can be negative**.
We do not change that; the results below flag any negative coefficient.

### Rate convention
Matches the Bayesian code: $2A \to \cdots$ has propensity $k\,A(A-1)/2$.
Their coefficient $c$ on the basis $A(A-1)$ therefore corresponds to $k = 2c$.
For $\emptyset$, $A$, $B$ and $AB$ reactants, $k = c$.

### Steps
1. Settings
2. Candidate mass-action reactions per channel (from `example4_network.json`)
3. Export trajectory and write config (`xx_basis_flag = 0`)
4. Run `prepare`
5. Restrict each channel's basis file (originals kept as `*_full.txt`)
6. Run `sparse_learning`
7. Results: LASSO rates vs. true rates vs. Bayesian posterior


## 1. Settings

In [1]:
import os, sys, re, glob, shutil, subprocess
import numpy as np
import pandas as pd

REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from src.parsing import export_for_sparse_learning, load_reaction_network

# Location of the compiled sparse-learning-CRN programs (prepare, sparse_learning).
# Override with the environment variable SPARSE_LEARNING_SRC if they live elsewhere.
SPARSE_LEARNING_SRC = os.environ.get(
    "SPARSE_LEARNING_SRC",
    "/Users/suzannesindi/Dropbox/WritingProjects/2026_06_08_CRNWork/sparse-learning-CRN/src",
)
PREPARE_EXE = os.path.join(SPARSE_LEARNING_SRC, "prepare")
SPARSE_EXE  = os.path.join(SPARSE_LEARNING_SRC, "sparse_learning")

NETWORK_FILE = os.path.join(REPO, "data", "example4_network.json")
TRAJ_FILE    = os.path.join(REPO, "data", "example4_T40_trajectory.json")
MCMC_FILE    = os.path.join(REPO, "results", "example4_T40", "mcmc_summary.xlsx")
STANDARD_RUN = os.path.join(REPO, "sparse_runs", "example4_T40_lam0p01")   # unrestricted run, for reference

T      = 40
LAMBDA = 0.01
lam_str = str(LAMBDA).replace(".", "p")
RUN_DIR = os.path.join(REPO, "sparse_runs", f"example4_T{T}_lam{lam_str}_massaction")

# Safety: never silently overwrite an existing run.
OVERWRITE = False

for p in [PREPARE_EXE, SPARSE_EXE, NETWORK_FILE, TRAJ_FILE]:
    print(("ok      " if os.path.exists(p) else "MISSING ") + p)
print("\nRun directory:", RUN_DIR)
if os.path.exists(RUN_DIR) and not OVERWRITE:
    raise RuntimeError(f"{RUN_DIR} already exists. Set OVERWRITE = True to redo the run "
                       "(or skip to section 7 to just read the results).")


ok      /home/claude/w/2026_06_08_CRNWork/sparse-learning-CRN/src/prepare
ok      /home/claude/w/2026_06_08_CRNWork/sparse-learning-CRN/src/sparse_learning
ok      /home/claude/w/BayesCRNInference/data/example4_network.json
ok      /home/claude/w/BayesCRNInference/data/example4_T40_trajectory.json

Run directory: /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction


## 2. Candidate mass-action reactions per channel

These come from the same reaction library as the Bayesian analysis (`compatible_reactions` in `example4_network.json`),
so the LASSO and the Bayesian method see exactly the same candidate reactions.

In [2]:
(rn4, _, rnames4, pnames4, trueTheta4, pvals4, sampled4,
 rmat4, uc4, cr4, sp4) = load_reaction_network(NETWORK_FILE)

true_k = {gi: pvals4[p] for gi, p in zip(sampled4, pnames4)}   # reaction index -> true rate

def side(v):
    parts = [s if int(n) == 1 else f"{int(n)}{s}" for n, s in zip(v, sp4) if int(n) != 0]
    return "+".join(parts) if parts else "∅"

def rxn_str(gi, change):
    reac = rmat4[:, gi]
    return f"{side(reac)} -> {side(reac + np.array(change))}"

# Channels that actually occur in the trajectory (the ones prepare will find)
CHANNELS = [(1, 0), (-1, 0), (-1, 1), (0, -1)]
for ch in CHANNELS:
    print(f"channel {ch}:")
    for gi in cr4[ch]:
        print(f"    {rxn_str(gi, ch):14s}  reactant={tuple(int(x) for x in rmat4[:, gi])}  true k = {true_k.get(gi, 0.0)}")


Reaction network loaded from /home/claude/w/BayesCRNInference/data/example4_network.json
  Species:   ['A', 'B']
  Reactions: 4 sampled  |  30 total in full CRN
  Unique stoichiometric changes: 18
channel (1, 0):
    ∅ -> A          reactant=(0, 0)  true k = 0.0
    A -> 2A         reactant=(1, 0)  true k = 1.0
    B -> A+B        reactant=(0, 1)  true k = 0.0
channel (-1, 0):
    A -> ∅          reactant=(1, 0)  true k = 0.1
    2A -> A         reactant=(2, 0)  true k = 0.0
    A+B -> B        reactant=(1, 1)  true k = 0.0
channel (-1, 1):
    A -> B          reactant=(1, 0)  true k = 0.0
    2A -> A+B       reactant=(2, 0)  true k = 0.0
    A+B -> 2B       reactant=(1, 1)  true k = 0.01
channel (0, -1):
    B -> ∅          reactant=(0, 1)  true k = 0.5
    2B -> B         reactant=(0, 2)  true k = 0.0
    A+B -> A        reactant=(1, 1)  true k = 0.0


## 3. Export trajectory and write config

`export_for_sparse_learning` writes the standard config with `xx_basis_flag = 1`.
We change that single line to `xx_basis_flag = 0` and print the result so the change is visible.

In [3]:
if os.path.exists(RUN_DIR) and OVERWRITE:
    shutil.rmtree(RUN_DIR)

export_for_sparse_learning(
    trajectory_files=[TRAJ_FILE],
    working_dir=RUN_DIR,
    T=T,
    poly_order=2,
    regular_lambda=LAMBDA,
    tot_step=20000,
)

cfg_path = os.path.join(RUN_DIR, "sparse_learning.cfg")
cfg = open(cfg_path).read()
assert re.search(r"^xx_basis_flag = 1 ;$", cfg, flags=re.M), "expected xx_basis_flag = 1 in exported config"
cfg = re.sub(r"^xx_basis_flag = 1 ;$",
             "# Mass-action run: quadratic basis is x(x-1), not x*x\nxx_basis_flag = 0 ;",
             cfg, flags=re.M)
open(cfg_path, "w").write(cfg)

print("\n--- active config lines ---")
print("\n".join(l for l in cfg.splitlines() if l.strip() and not l.startswith("#")))


Written: /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction/traj_data/traj_0.txt  (5804 states, 2 species)

Working directory ready: /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction
  Trajectories : 1  (2 species each)
  Config       : /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction/sparse_learning.cfg

Next steps (run from your shell):
  cd /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction
  /path/to/sparse-learning-CRN/src/prepare
  /path/to/sparse-learning-CRN/src/sparse_learning

Results will appear in /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction/output/

--- active config lines ---
T = 40.0 ;
N_traj = 1 ;
poly_order = 2 ;
regular_lambda = 0.01 ;
solver_id = 1 ;
flag_backtracking = 1 ;
tot_step = 20000 ;
cost_stop_tol = 1e-07 ;
epsL1_flag = 0 ;
l1_eps = 0.01 ;
xx_basis_flag = 0 ;
eps = 0.1 ;
g_cut = 35.0 ;
max_step_since_prev_min_cost = 1000 

## 4. Run `prepare`

This finds the reaction channels in the trajectory and writes one basis file per channel (all basis functions switched on).

In [4]:
def run(exe, cwd):
    print(f"$ cd {cwd}\n$ {exe}\n")
    p = subprocess.run([exe], cwd=cwd, capture_output=True, text=True)
    out = p.stdout.strip().splitlines()
    print("\n".join(out[-25:]))          # last lines of the program's own output
    if p.returncode != 0:
        print(p.stderr)
        raise RuntimeError(f"{os.path.basename(exe)} exited with code {p.returncode}")
    print(f"\n[{os.path.basename(exe)} finished, exit code 0]")

run(PREPARE_EXE, RUN_DIR)


$ cd /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction
$ /home/claude/w/2026_06_08_CRNWork/sparse-learning-CRN/src/prepare


Basis functions are written into the file: ./output/basis_funct_info.txt

Basis info of channel 0 is written to : ./output/basis_of_channel_0.txt
Range of each basis function in channel 0: 
[1.00e+00, 1.00e+00]	[2.00e+00, 2.14e+02]	[1.40e+01, 2.71e+02]	[2.00e+00, 4.56e+04]	[1.40e+02, 3.22e+04]	[1.82e+02, 7.32e+04]	

Basis info of channel 1 is written to : ./output/basis_of_channel_1.txt
Range of each basis function in channel 1: 
[1.00e+00, 1.00e+00]	[1.00e+00, 2.15e+02]	[9.00e+00, 2.80e+02]	[0.00e+00, 4.60e+04]	[6.40e+01, 3.23e+04]	[7.20e+01, 7.81e+04]	

Basis info of channel 2 is written to : ./output/basis_of_channel_2.txt
Range of each basis function in channel 2: 
[1.00e+00, 1.00e+00]	[0.00e+00, 2.09e+02]	[1.00e+01, 2.81e+02]	[0.00e+00, 4.35e+04]	[0.00e+00, 3.19e+04]	[9.00e+01, 7.87e+04]	

Basis info of channel 3 is written to : ./

In [5]:
OUT = os.path.join(RUN_DIR, "output")

def read_channel_info(out_dir):
    tok = open(os.path.join(out_dir, "channel_info.txt")).read().split()
    n_ch, n_sp = int(tok[0]), int(tok[1])
    vals = list(map(int, tok[2:]))
    changes = [tuple(vals[i*n_sp:(i+1)*n_sp]) for i in range(n_ch)]
    counts = vals[n_ch*n_sp:n_ch*n_sp + n_ch]
    return changes, counts

def read_basis_info(out_dir):
    """Basis functions as tuples of species indices: () = 1, (0,) = A, (0,1) = AB, ..."""
    tok = list(map(int, open(os.path.join(out_dir, "basis_funct_info.txt")).read().split()))
    n_sp, n_b = tok[0], tok[1]
    basis, pos = [], 2
    for _ in range(n_b):
        k = tok[pos]; basis.append(tuple(tok[pos+1:pos+1+k])); pos += 1 + k
    return basis

def basis_label(b, xx_flag):
    if len(b) == 0: return "1"
    if len(b) == 1: return sp4[b[0]]
    i, j = b
    if i == j: return f"{sp4[i]}({sp4[i]}-1)" if xx_flag == 0 else f"{sp4[i]}^2"
    return sp4[i] + sp4[j]

ch_changes, ch_counts = read_channel_info(OUT)
basis = read_basis_info(OUT)
print("Basis functions (xx_basis_flag = 0):")
for i, b in enumerate(basis):
    print(f"  {i}: {basis_label(b, 0)}")
print("\nChannels found by prepare:")
for i, (c, n) in enumerate(zip(ch_changes, ch_counts)):
    print(f"  channel {i}: change {c}, {n} events")
assert sorted(ch_changes) == sorted(CHANNELS), "unexpected channels"


Basis functions (xx_basis_flag = 0):
  0: 1
  1: A
  2: B
  3: A(A-1)
  4: AB
  5: B(B-1)

Channels found by prepare:
  channel 0: change (-1, 0), 190 events
  channel 1: change (-1, 1), 1843 events
  channel 2: change (0, -1), 1837 events
  channel 3: change (1, 0), 1933 events


## 5. Restrict each channel's basis file to mass-action terms

For each channel, a candidate reaction with reactant vector $r$ uses the basis function
built from that reactant ($\emptyset\to 1$, $A\to A$, $2A\to A(A-1)$, $A+B\to AB$, ...).
Only those basis functions are kept. The original file is saved as `basis_of_channel_<i>_full.txt`.

File format (theirs): number of basis functions; their indices; a weight per basis function;
an initial value per basis function; then the min and max of each basis function along the trajectory.

In [6]:
def reactant_to_basis_index(reac):
    key = tuple(sorted(s for s, n in enumerate(reac) for _ in range(int(n))))
    if key not in basis:
        raise ValueError(f"reactant {reac} has no matching basis function")
    return basis.index(key)

def read_basis_file(path):
    tok = open(path).read().split()
    n = int(tok[0]); v = tok[1:]
    fields = [v[k*n:(k+1)*n] for k in range(len(v) // n)]   # idx, weight, init, (min, max)
    return n, fields

KEEP = {}   # channel number -> list of (basis index, reaction index)
for ch_i, change in enumerate(ch_changes):
    path = os.path.join(OUT, f"basis_of_channel_{ch_i}.txt")
    full = path.replace(".txt", "_full.txt")
    shutil.copy2(path, full)

    n, fields = read_basis_file(full)
    idx_full = [int(x) for x in fields[0]]
    keep = [(reactant_to_basis_index(rmat4[:, gi]), gi) for gi in cr4[change]]
    pos = [idx_full.index(b) for b, _ in keep]
    KEEP[ch_i] = keep

    with open(path, "w") as f:
        f.write(f"{len(pos)} \n")
        for row in fields:
            f.write(" ".join(row[p] for p in pos) + " \n")
            if row is fields[2]:
                f.write("\n")

    print(f"channel {ch_i} {change}:  {n} basis functions -> {len(pos)}")
    print("    before:", [basis_label(basis[b], 0) for b in idx_full])
    print("    after: ", [basis_label(basis[b], 0) for b, _ in keep],
          " <->  ", [rxn_str(gi, change) for _, gi in keep])
    print("    file:  ", open(path).read().replace("\n", " | "))


channel 0 (-1, 0):  6 basis functions -> 3
    before: ['1', 'A', 'B', 'A(A-1)', 'AB', 'B(B-1)']
    after:  ['A', 'A(A-1)', 'AB']  <->   ['A -> ∅', '2A -> A', 'A+B -> B']
    file:   3  | 1 3 4  | 1 1 1  | 0 0 0  |  | 2 2 140  | 214 45582 32224  | 
channel 1 (-1, 1):  6 basis functions -> 3
    before: ['1', 'A', 'B', 'A(A-1)', 'AB', 'B(B-1)']
    after:  ['A', 'A(A-1)', 'AB']  <->   ['A -> B', '2A -> A+B', 'A+B -> 2B']
    file:   3  | 1 3 4  | 1 1 1  | 0 0 0  |  | 1 0 64  | 215 46010 32342  | 
channel 2 (0, -1):  6 basis functions -> 3
    before: ['1', 'A', 'B', 'A(A-1)', 'AB', 'B(B-1)']
    after:  ['B', 'B(B-1)', 'AB']  <->   ['B -> ∅', '2B -> B', 'A+B -> A']
    file:   3  | 2 5 4  | 1 1 1  | 0 0 0  |  | 10 90 0  | 281 78680 31930  | 
channel 3 (1, 0):  6 basis functions -> 3
    before: ['1', 'A', 'B', 'A(A-1)', 'AB', 'B(B-1)']
    after:  ['1', 'A', 'B']  <->   ['∅ -> A', 'A -> 2A', 'B -> A+B']
    file:   3  | 0 1 2  | 1 1 1  | 0 0 0  |  | 1 1 9  | 1 214 280  | 


## 6. Run `sparse_learning`

After it finishes, the termination reason for each channel is read from their log, so a channel that stopped without converging is visible.

In [7]:
run(SPARSE_EXE, RUN_DIR)


$ cd /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction
$ /home/claude/w/2026_06_08_CRNWork/sparse-learning-CRN/src/sparse_learning



Solving coefficients for the reaction channel 3... finished.

	Reason of termination:	Changes of the costs in the last 20 steps are under threshold.

	Total steps=1548,	 Final cost = -1.61174856e+02,	 min_cost=-1.61175275e+02 (at Step 1143)

	Rel. residual of vec. = 1.477253e-02 
	Rel. diff. of cost = 1.763231e-08
	Rel. diff. of cost (w.r.t. optimal) = 2.577430e-06

	Costs in the last 20 steps: [-1.611749e+02, -1.611748e+02],	Rel-diff. = 9.6619e-08 (1.00e-07)

	Results are written to: ./output/omega_vec_for_channel_3.txt
	Corresponding cost : -1.611753e+02

	Coeff. of Channel 3 (1 0):
		1.9278467759454773e-04	1.0041015916360951e+00	-1.2550420624049953e-03	

Final cost of all channels = -468.654482



Runtime : 13.63 sec.

[sparse_learning finished, exit code 0]


In [8]:
def termination_summary(run_dir):
    log_path = os.path.join(run_dir, "log", "sparse_infer.log")
    if not os.path.exists(log_path):
        return f"(no log saved in {os.path.relpath(run_dir, REPO)})"
    log = open(log_path).read()
    blocks = re.split(r"Solving coefficients for the reaction channel (\d+)\.\.\. finished\.", log)
    rows = []
    for k in range(1, len(blocks), 2):
        ch, body = int(blocks[k]), blocks[k+1]
        reason = re.search(r"Reason of termination:\s*(.*)", body)
        steps  = re.search(r"Total steps=(\d+),\s*Final cost = (\S+),\s*min_cost=(\S+) \(at Step (\d+)\)", body)
        rows.append(dict(channel=ch, change=ch_changes[ch],
                         reason=reason.group(1).strip() if reason else "?",
                         total_steps=int(steps.group(1)) if steps else None,
                         best_at_step=int(steps.group(4)) if steps else None))
    return pd.DataFrame(rows)

print("Mass-action-restricted run:")
display(termination_summary(RUN_DIR))
print("\nStandard (unrestricted) run, for reference:")
display(termination_summary(STANDARD_RUN))
print("Note: when the reason is 'Minimal cost hasn't been updated...', the program stopped without meeting "
      "its convergence tolerance and reports the lowest-cost iterate it saw.")


Mass-action-restricted run:


,channel,change,reason,total_steps,best_at_step
0,0,"(-1, 0)",Minimal cost hasn't been updated after 1000 st...,2958,1957
1,1,"(-1, 1)",Changes of the costs in the last 20 steps are ...,47,47
2,2,"(0, -1)",Changes of the costs in the last 20 steps are ...,2515,1525
3,3,"(1, 0)",Changes of the costs in the last 20 steps are ...,1548,1143



Standard (unrestricted) run, for reference:


'(no log saved in sparse_runs/example4_T40_lam0p01)'

Note: when the reason is 'Minimal cost hasn't been updated...', the program stopped without meeting its convergence tolerance and reports the lowest-cost iterate it saw.


## 7. Results

Coefficients are read from `output/omega_vec_for_channel_<i>.txt` (one value per kept basis function, same order as the basis file).
Rates use the Bayesian convention: $k = 2c$ for $2A$ and $2B$ reactants, $k = c$ otherwise.

In [9]:
def read_omega(out_dir, ch_i):
    tok = open(os.path.join(out_dir, f"omega_vec_for_channel_{ch_i}.txt")).read().split()
    return np.array(list(map(float, tok[1:1 + int(tok[0])])))

def rate_factor(reac):
    return 2.0 if any(int(n) == 2 for n in reac) else 1.0   # k A(A-1)/2  <->  c A(A-1)

# Bayesian posterior summaries (same channel mapping as the paper's predator-prey table)
df4 = pd.read_excel(MCMC_FILE)
count_to_ch = {n: c for c, n in zip(ch_changes, ch_counts)}
run_to_ch = {int(ri): count_to_ch.get(int(g["Count"].iloc[0])) for ri, g in df4.groupby("Run_Index")}
mcmc = {(run_to_ch[int(r["Run_Index"])], int(r["Param_Index"])): r
        for _, r in df4.iterrows() if run_to_ch.get(int(r["Run_Index"])) is not None}

# Unrestricted run as a polynomial string, for reference
std_changes, _ = read_channel_info(os.path.join(STANDARD_RUN, "output"))
std_basis = read_basis_info(os.path.join(STANDARD_RUN, "output"))
def std_poly(change, thr=1e-3):
    i = std_changes.index(change)
    w = read_omega(os.path.join(STANDARD_RUN, "output"), i)
    return " ".join(f"{v:+.4f}·{basis_label(b, 1)}" for v, b in zip(w, std_basis) if abs(v) >= thr)

rows = []
for ch_i, change in enumerate(ch_changes):
    c = read_omega(OUT, ch_i)
    for pi, ((b, gi), coef) in enumerate(zip(KEEP[ch_i], c)):
        reac = rmat4[:, gi]
        k_lasso = rate_factor(reac) * coef
        m = mcmc.get((change, cr4[change].index(gi)))
        rows.append({
            "channel": str(change),
            "reaction": rxn_str(gi, change),
            "basis": basis_label(basis[b], 0),
            "true k": true_k.get(gi, 0.0),
            "LASSO coef c": coef,
            "LASSO k": k_lasso,
            "negative?": "NEGATIVE" if k_lasso < 0 else "",
            "Bayes mean": None if m is None else m["Mean"],
            "Bayes 95% CI": None if m is None else f"({m['CI_lower']:.4f}, {m['CI_upper']:.4f})",
            "Pr(off)": None if m is None else m["Prob_Off"],
        })
res = pd.DataFrame(rows)
order = {str(c): i for i, c in enumerate(CHANNELS)}
res = res.sort_values("channel", key=lambda s: s.map(order), kind="stable").reset_index(drop=True)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
display(res)

res.to_csv(os.path.join(RUN_DIR, "massaction_lasso_results.csv"), index=False)
print("Saved:", os.path.join(RUN_DIR, "massaction_lasso_results.csv"))


,channel,reaction,basis,true k,LASSO coef c,LASSO k,negative?,Bayes mean,Bayes 95% CI,Pr(off)
0,"(1, 0)",∅ -> A,1,0.0000,0.0002,0.0002,,0.0045,"(0.0000, 0.0000)",0.9947
1,"(1, 0)",A -> 2A,A,1.0000,1.0041,1.0041,,1.0024,"(0.9587, 1.0480)",0.0000
2,"(1, 0)",B -> A+B,B,0.0000,-0.0013,-0.0013,NEGATIVE,0.0000,"(0.0000, 0.0000)",1.0000
3,"(-1, 0)",A -> ∅,A,0.1000,0.1097,0.1097,,0.0995,"(0.0859, 0.1140)",0.0000
4,"(-1, 0)",2A -> A,A(A-1),0.0000,-0.0001,-0.0003,NEGATIVE,0.0000,"(0.0000, 0.0000)",1.0000
5,"(-1, 0)",A+B -> B,AB,0.0000,0.0000,0.0000,,0.0000,"(0.0000, 0.0000)",1.0000
6,"(-1, 1)",A -> B,A,0.0000,0.0001,0.0001,,0.0000,"(0.0000, 0.0000)",1.0000
7,"(-1, 1)",2A -> A+B,A(A-1),0.0000,0.0001,0.0001,,0.0000,"(0.0000, 0.0000)",1.0000
8,"(-1, 1)",A+B -> 2B,AB,0.0100,0.0100,0.0100,,0.0101,"(0.0096, 0.0105)",0.0000
9,"(0, -1)",B -> ∅,B,0.5000,0.4782,0.4782,,0.4915,"(0.4695, 0.5136)",0.0000


Saved: /home/claude/w/BayesCRNInference/sparse_runs/example4_T40_lam0p01_massaction/massaction_lasso_results.csv


In [10]:
print("Unrestricted LASSO (standard run, xx_basis_flag = 1, all 6 basis functions), terms with |c| >= 0.001:\n")
for ch in CHANNELS:
    print(f"  {str(ch):9s}  {std_poly(ch)}")


Unrestricted LASSO (standard run, xx_basis_flag = 1, all 6 basis functions), terms with |c| >= 0.001:

  (1, 0)     +0.0242·1 +0.9687·A +0.0011·B
  (-1, 0)    +0.0010·1 +0.0181·A +0.0361·B
  (-1, 1)    +0.0075·B +0.0102·AB
  (0, -1)    -0.0012·1 -0.0381·A +0.4970·B


## 8. LaTeX table for the paper

Written to `writing/table_predprey_detail_massaction.tex`; the existing `table_predprey_detail.tex` is not touched.
Every number is read from files: true rates from `example4_network.json`, Bayesian summaries from `mcmc_summary.xlsx`,
LASSO values from the two `sparse_runs` folders. Nothing is typed in by hand.

Unrestricted LASSO entries list all terms with $|c| \ge 0.001$; $^\dagger$ marks an entry containing a term that no
candidate mass-action reaction in that channel can produce, or a negative term.

In [11]:
TEX_OUT = os.path.join(REPO, "writing", "table_predprey_detail_massaction.tex")

def tex_side(v):
    parts = [s if int(n) == 1 else f"{int(n)}{s}" for n, s in zip(v, sp4) if int(n) != 0]
    return "+".join(parts) if parts else r"\emptyset"

def tex_rxn(gi, change):
    reac = rmat4[:, gi]
    return f"${tex_side(reac)}\\to {tex_side(reac + np.array(change))}$"

def tex_num(v, nd=4):
    s = f"{v:.{nd}f}"
    if s.startswith("-"):
        s = "-" + s[1:] if float(s) != 0 else s[1:]
    return f"${s}$"

def tex_basis(b):          # unrestricted run uses xx_basis_flag = 1 (x*x)
    if len(b) == 0: return ""
    if len(b) == 1: return sp4[b[0]]
    i, j = b
    return f"{sp4[i]}^2" if i == j else sp4[i] + sp4[j]

def tex_std_poly(change, thr=1e-3):
    i = std_changes.index(change)
    w = read_omega(os.path.join(STANDARD_RUN, "output"), i)
    allowed = {tuple(sorted(s for s, n in enumerate(rmat4[:, gi]) for _ in range(int(n)))) for gi in cr4[change]}
    terms, bad = [], False
    for v, b in zip(w, std_basis):
        if abs(v) < thr: continue
        if b not in allowed or v < 0: bad = True
        lab = tex_basis(b)
        mag = f"{abs(v):.3f}" + (f"\\,{lab}" if lab else "")
        terms.append(("-" if v < 0 else "+", mag))
    s = " ".join(f"{sg} {m}" for sg, m in terms)
    s = s[2:] if s.startswith("+ ") else "-" + s[2:]
    return f"${s}{'^{\\,\\dagger}' if bad else ''}$"

lasso_k = {(r["channel"], r["reaction"]): r["LASSO k"] for _, r in res.iterrows()}
n_events = dict(zip(ch_changes, ch_counts))
chlabel = {c: f"$({c[0]:+d},\\;{c[1]:+d})$".replace("+0", "0") for c in CHANNELS}

L = []
L.append(r"% Predator-prey detail with mass-action-restricted LASSO")
L.append(r"% Generated by examples/example4_sparse_massaction.ipynb -- do not edit by hand")
L.append(r"\begin{table}[htbp]")
L.append(r"\centering")
L.append(r"\small")
L.append(r"\setlength{\tabcolsep}{4pt}")
L.append(r"\caption{Predator-prey inference at $T=" + f"{T}" + r"$ (" + f"{sum(ch_counts):,}".replace(",", "{,}") +
         r" events). All candidate mass-action reactions for each stoichiometric channel $\xi_\ell$ are shown. "
         r"$k^*$: true rate (bold if active). "
         r"$\hat{k}$: posterior mean (95\% credible interval); ``---'' when $\Pr(\text{off})>0.99$. "
         r"$\Pr(\text{off})$: spike-and-slab exclusion probability. "
         r"LASSO (mass action): sparse-learning-CRN at $\lambda=" + f"{LAMBDA}" + r"$ with each channel restricted to the "
         r"basis functions of its candidate reactions ($1, x, x(x-1), xy$), reported as rate constants "
         r"($k=2c$ for $2X$ reactants). "
         r"LASSO (unrestricted): the same method with all basis functions $1, A, B, A^2, AB, B^2$; terms with "
         r"$|c|\ge 0.001$ shown. "
         r"$^\dagger$~contains a term not produced by any mass-action reaction in that channel, or a negative term.}")
L.append(r"\label{tab:predprey_detail_massaction}")
L.append(r"\begin{tabular}{l l r r l r r l}")
L.append(r"\toprule")
L.append(r"$\xi_\ell$ & Reaction & $k^*$ & Events & $\hat{k}$\;(95\% CI) & $\Pr(\text{off})$ & "
         r"\shortstack{LASSO\\(mass action)} & \shortstack{LASSO\\(unrestricted)} \\")
L.append(r"\midrule")
for ch in CHANNELS:
    cands = cr4[ch]
    for pi, gi in enumerate(cands):
        tk = true_k.get(gi, 0.0)
        m = mcmc.get((ch, pi))
        if m is None:
            post, poff = "---", "---"
        else:
            poff = f"${m['Prob_Off']:.3f}$"
            post = "---" if m["Prob_Off"] >= 0.99 else \
                   f"${m['Mean']:.4f}\\;({m['CI_lower']:.4f},\\,{m['CI_upper']:.4f})$"
        cols = [
            f"\\multirow{{{len(cands)}}}{{*}}{{{chlabel[ch]}}}" if pi == 0 else "",
            tex_rxn(gi, ch),
            f"\\textbf{{{tk:.4f}}}" if tk > 0 else "$0$",
            str(n_events[ch]) if pi == 0 else "",
            post, poff,
            tex_num(lasso_k[(str(ch), rxn_str(gi, ch))]),
            tex_std_poly(ch) if pi == 0 else "",
        ]
        L.append(" & ".join(cols) + r" \\")
    L.append(r"\midrule")
L[-1] = r"\bottomrule"
L.append(r"\end{tabular}")
L.append(r"\end{table}")

open(TEX_OUT, "w").write("\n".join(L) + "\n")
print("Saved:", TEX_OUT, "\n")
print("\n".join(L))


Saved: /home/claude/w/BayesCRNInference/writing/table_predprey_detail_massaction.tex 

% Predator-prey detail with mass-action-restricted LASSO
% Generated by examples/example4_sparse_massaction.ipynb -- do not edit by hand
\begin{table}[htbp]
\centering
\small
\setlength{\tabcolsep}{4pt}
\caption{Predator-prey inference at $T=40$ (5{,}803 events). All candidate mass-action reactions for each stoichiometric channel $\xi_\ell$ are shown. $k^*$: true rate (bold if active). $\hat{k}$: posterior mean (95\% credible interval); ``---'' when $\Pr(\text{off})>0.99$. $\Pr(\text{off})$: spike-and-slab exclusion probability. LASSO (mass action): sparse-learning-CRN at $\lambda=0.01$ with each channel restricted to the basis functions of its candidate reactions ($1, x, x(x-1), xy$), reported as rate constants ($k=2c$ for $2X$ reactants). LASSO (unrestricted): the same method with all basis functions $1, A, B, A^2, AB, B^2$; terms with $|c|\ge 0.001$ shown. $^\dagger$~contains a term not produced b